In [1]:
 !pip install -qU langchain langchain-community langchain-huggingface langchain-google-genai chromadb rank_bm25 datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/2

In [2]:
import pandas as pd
import numpy as np
import re
import html
from datasets import load_dataset
from langchain_community.document_loaders import DataFrameLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from tqdm import tqdm
print("Done")

/tmp/ipykernel_1987/1480620507.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DataFrameLoader


Done


In [3]:
#loading dataset online
dataset = load_dataset("luisroque/instruct-python-500k")
print(dataset)

README.md:   0%|          | 0.00/1.82k [00:00<?, ?B/s]

data/train-00000-of-00002-4e61d830cfab16(…): reconstructing file:   0%|          |  0.00B /  262MB            

data/train-00000-of-00002-4e61d830cfab16(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-4904c95133ba31(…): reconstructing file:   0%|          |  0.00B /  288MB            

data/train-00001-of-00002-4904c95133ba31(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/501349 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['score_question', 'score_answer', 'question', 'answer', '__index_level_0__'],
        num_rows: 501349
    })
})


In [4]:
# convert raw data into dataframe and take small sample
df_full = dataset["train"].to_pandas()
df = df_full.sample(n=100000, random_state=42).reset_index(drop=True)
df.head()

,score_question,score_answer,question,answer,__index_level_0__
0,1,5,Reverse each iterable in a list using function...,"If you're content to work only with sequences,...",772112
1,13,11,Errno 10061 : No connection could be made beca...,"10061 is WSAECONNREFUSED, 'connection refused'...",313671
2,20,12,Improving OCR performance on multi-paragraph s...,Tesseract is very good on clean input text (li...,284709
3,1,3,Removing all float with the range of .01 to .9...,"""I have a simple python program""\nThis is not ...",341439
4,0,0,How to append elements to a list by using redu...,Here's a possible solution without using reduc...,960136


In [5]:
# Cleans raw text by decoding HTML entities, stripping tags, and normalizing whitespace.
def clean_html_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)                           # amp& ===> &
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

In [6]:
before = len(df)

df["question_clean"] = df["question"].apply(clean_html_text)
df["answer_clean"] = df["answer"].apply(clean_html_text)

df = df[(df["question_clean"].str.len() > 10) & (df["answer_clean"].str.len() > 10)].reset_index(drop=True)

print(f"[diag] rows before: {before}, after cleaning: {len(df)}")

documents = [Document(page_content=row.question_clean,metadata={"answer": row.answer_clean,
                      "score_answer": int(row.score_answer)})
                       for row in df.itertuples()]

[diag] rows before: 100000, after cleaning: 99991


In [7]:
df[["question_clean", "answer_clean"]].head(5)

,question_clean,answer_clean
0,Reverse each iterable in a list using function...,"If you're content to work only with sequences,..."
1,Errno 10061 : No connection could be made beca...,"10061 is WSAECONNREFUSED, 'connection refused'..."
2,Improving OCR performance on multi-paragraph s...,Tesseract is very good on clean input text (li...
3,Removing all float with the range of .01 to .9...,"""I have a simple python program"" This is not a..."
4,How to append elements to a list by using redu...,Here's a possible solution without using reduc...


In [8]:
"""
               Raw Dataset (instruct-python-500k)
                                │
                                ▼
                   Sampling Subset (100,000 Rows)
                                │
                                ▼
                       Text Preprocessing
              (HTML Unescaping + Regex Cleaning)
                                │
                                ▼
                 Document Object Transformation
           (page_content = Question, metadata = Answer)
                                │
                                │
                 ┌───────────────────────────────┐
                 │                               │
                 ▼                               ▼
     ChromaDB Vector Store               BM25 Index Creation
    (Sentence-Transformers MiniLM)      (Tokenized Terms Index)
                 │                               │
                 ▼                               ▼
    Dense Retriever (k=10)           Sparse Retriever (k=10)
                 │                               │
                 └───────────────┬───────────────┘
                                 │
                                 ▼
                     EnsembleRetriever (RRF)


"""

'\n               Raw Dataset (instruct-python-500k)\n                                │\n                                ▼\n                   Sampling Subset (100,000 Rows)\n                                │\n                                ▼\n                       Text Preprocessing\n              (HTML Unescaping + Regex Cleaning)\n                                │\n                                ▼\n                 Document Object Transformation\n           (page_content = Question, metadata = Answer)\n                                │\n                                │\n                 ┌───────────────────────────────┐\n                 │                               │\n                 ▼                               ▼\n     ChromaDB Vector Store               BM25 Index Creation\n    (Sentence-Transformers MiniLM)      (Tokenized Terms Index)\n                 │                               │\n                 ▼                               ▼\n    Dense Retriever (k=10)   

In [9]:
#Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#Chroma VectorStore (Dense Search)
vectorstore = Chroma.from_documents(documents=documents,embedding=embeddings,
                                    collection_name="so_qa_langchain",
                                    persist_directory="./chroma_langchain")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
#the dense retrieval(cosin similarity search)
chroma_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

#the bm25 retrieval(keyword-search)
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 10

# Combines keyword-based (BM25) and vector-based (Chroma) search using equal weighting for hybrid retrieval
hybrid_retriever = EnsembleRetriever(retrievers=[bm25_retriever, chroma_retriever],
                                     weights=[0.5, 0.5])

# Executes hybrid retrieval to fetch the top 5 most relevant chunks
results = hybrid_retriever.invoke("your query", config={"configurable": {"k": 5}})
print("Hybrid Retriever built successfully!")

Hybrid Retriever built successfully!


In [11]:
import os
from collections import deque

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from getpass import getpass

In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.2,
    do_sample=True
)

llm = HuggingFacePipeline(pipeline=pipe)

Enter your Google API Key: ··········


In [13]:
qa_system_prompt = """You are a Python coding assistant. A user asked a question.
Below are similar Q&A pairs retrieved from a knowledge base for reference context only.
They may not match the user's exact situation - adapt the reasoning/solution to fit
the user's specific question rather than copying an answer verbatim.

Instructions:
- If the reference context is directly relevant, use its approach but adapt it to the user's exact code/variables/situation.
- If the reference context only partially applies, explain what's different and adjust the solution accordingly.
- If none of the context is relevant enough, say so and answer from general Python knowledge.
- Always give a concrete, runnable code answer when applicable.

Reference context (similar past Q&A, for guidance only):
{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),

    MessagesPlaceholder(variable_name="chat_history"), # saving Memory
    ("human", "{input}"),])



In [14]:
# mix all Hybrid Retrieved chunks into one context
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(f"[{i}] Q: {doc.page_content}\n    A: {doc.metadata['answer']}")
    return "\n\n".join(formatted)

# chain core - add the prompt and the LLM and the output into on chain
rag_chain = (RunnablePassthrough.assign(
context=lambda x: format_docs(hybrid_retriever.invoke(x["input"])))
    | qa_prompt
    | llm
    | StrOutputParser())


In [15]:
# Custom Bounded Memory (Sliding Window)

class Last5ChatMessageHistory(BaseChatMessageHistory):
    def __init__(self, max_messages=10):
        self.max_messages = max_messages
        self._messages = deque(maxlen=max_messages)

    @property
    def messages(self) -> list[BaseMessage]:
        return list(self._messages)

    def add_message(self, message: BaseMessage):
        self._messages.append(message)

    def clear(self):
        self._messages.clear()


store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = Last5ChatMessageHistory(max_messages=10)

    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,get_session_history,
    input_messages_key="input",history_messages_key="chat_history",
    output_messages_key="output")


/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [20]:
"""

                           User Input + session_id
                                      │
                                      ▼
                        RunnableWithMessageHistory
                                      │
                                      ▼
                      Fetch Chat History from Store
                            (store[session_id])
                                      │
                                      ▼
                         RunnablePassthrough.assign
                                      │
                                      ▼
                       Trigger Hybrid Search (Invoke)
                                      │
                       ┌──────────────┴──────────────┐
                       │                             │
                       ▼                             ▼
            BM25 Sparse Search            Chroma Vector Search
             (Top 10 Matches)              (Top 10 Matches)
                       │                             │
                       └──────────────┬──────────────┘
                                      │
                                      ▼
                        Reciprocal Rank Fusion (RRF)
                        (Select & Re-rank Top 5)
                                      │
                                      ▼
                             Format Docs Context
                         ([1] Q: ... A: ... Text)
                                      │
                                      ▼
                           Construct Chat Prompt
           ┌──────────────────────────────────────────────────┐
           │ System Message (Strict Rules + Retrived Context) │
           │ MessagesPlaceholder (Inject Chat History)        │
           │ Human Message (User Input)                       │
           └──────────────────────────────────────────────────┘
                                      │
                                      ▼
                          ChatGoogleGenerativeAI
                          (Qwen/Qwen2.5-7B-Instruct Local GPU)
                                      │
                                      ▼
                           StrOutputParser Engine
                          (Extract Plain Response)
                                      │
                                      │
                                      ▼

                Update Chat History       Return Final Response
             (Append Query & Answer)  +     to User Client


"""

'\n\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0User Input + session_id\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 │\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 ▼\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 RunnableWithMessageHistory\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 │\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 ▼\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 Fetch Chat History from Store\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 (store[session_id])\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 │\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 ▼\n\xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa

In [17]:
session_config = {"configurable": {"session_id": "user_session_1"}}

print("Python_RAG_Assistant(PyRexa)")
print("Type 'exit' to end the conversation.\n")

while True:
    query = input("You: ")

    if query.lower().strip() == "exit":
        print("Conversation ended.")
        break

    response = conversational_rag_chain.invoke({"input": query},config=session_config)

    print("\nAssistant:")
    print(response)
    print()

Python_RAG_Assistant(PyRexa)
Type 'exit' to end the conversation.

You: How To Reverse Frpm List To Dict



Assistant:
Because "reversing from a list to a dictionary" can mean a few different things in Python, here are the solutions for the most common scenarios.

---

### Scenario 1: You have a list of key-value pairs (tuples) and want to convert them into a dictionary in reverse order
If you have a list of pairs like `[('a', 1), ('b', 2)]` and want to build a dictionary starting from the last element:

```python
pairs = [('a', 1), ('b', 2), ('c', 3)]

# Use reversed() to reverse the list, then convert to a dict
reversed_dict = dict(reversed(pairs))

print(reversed_dict)
# Output: {'c': 3, 'b': 2, 'a': 1}
```

---

### Scenario 2: You have a dictionary with lists as values, and you want to "invert" it (list elements become keys)
If you have a dictionary like `{'a': [1, 2]}` and want to reverse it so the list elements become the keys: `{1: 'a', 2: 'a'}`:

```python
original_dict = {
    'group_A': ['apple', 'banana'],
    'group_B': ['cherry', 'date']
}

# Use a nested dictionary comprehens

In [18]:
history = get_session_history("user_session_1")

print("Number of stored messages:", len(history.messages))

for i, message in enumerate(history.messages, 1):
    print(f"{i}. {message.type}: {message.content[:-1]}")

Number of stored messages: 2
1. human: How To Reverse Frpm List To Dic
2. ai: Because "reversing from a list to a dictionary" can mean a few different things in Python, here are the solutions for the most common scenarios.

---

### Scenario 1: You have a list of key-value pairs (tuples) and want to convert them into a dictionary in reverse order
If you have a list of pairs like `[('a', 1), ('b', 2)]` and want to build a dictionary starting from the last element:

```python
pairs = [('a', 1), ('b', 2), ('c', 3)]

# Use reversed() to reverse the list, then convert to a dict
reversed_dict = dict(reversed(pairs))

print(reversed_dict)
# Output: {'c': 3, 'b': 2, 'a': 1}
```

---

### Scenario 2: You have a dictionary with lists as values, and you want to "invert" it (list elements become keys)
If you have a dictionary like `{'a': [1, 2]}` and want to reverse it so the list elements become the keys: `{1: 'a', 2: 'a'}`:

```python
original_dict = {
    'group_A': ['apple', 'banana'],
    'gr

In [ ]:
# ============================================================================
#  PYREXA — Premium Gradio UI (connected to your RAG pipeline)
#  Paste this in a NEW cell AFTER all the RAG cells (hybrid_retriever, qa_prompt,
#  llm, get_session_history, format_docs must already be defined) and run it.
#
#  Logo/avatar: paste your ORIGINAL two lines (LOGO_B64 / AVATAR_B64) in the marked
#  spot of the "assets" section below. They are used byte-for-byte, nothing is re-encoded.
# ============================================================================
import sys, subprocess, os, json, time, uuid, base64, inspect
from urllib.parse import quote

try:
    import gradio as gr
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=6.0,<7.0"])
    import gradio as gr

# ---------------------------------------------------------------- settings --
TOP_K = 5                        # number of retrieved chunks sent to the LLM
HISTORY_FILE = "pyrexa_chats.json"   # chat history is saved here (survives restarts)

# ------------------------------------------------------------------- assets --
# >>> PASTE YOUR ORIGINAL TWO LINES HERE, exactly as they were in your old code:
# >>>     LOGO_B64 = "iVBORw0KGgo........"
# >>>     AVATAR_B64 = "iVBORw0KGgo........"
# >>> If you already ran your old UI cell in this notebook session, you can leave this
# >>> empty: the values are still in memory and are reused automatically.

# ------------------------------------------------------------------- assets --
LOGO_B64 = "..."
AVATAR_B64 = "..."
LOGO_DATA = "data:image/png;base64," + LOGO_B64
AVATAR_DATA = "data:image/png;base64," + AVATAR_B64
AVATAR_PATH = os.path.abspath("pyrexa_avatar.png")
with open(AVATAR_PATH, "wb") as _f:
    _f.write(base64.b64decode(AVATAR_B64))

# ---------------------------------------------------------- RAG integration --
_ui_chain = None

def _pipeline_ready():
    return all(n in globals() for n in ("hybrid_retriever", "qa_prompt", "llm", "get_session_history", "format_docs"))

def _get_chain():
    """Same prompt + LLM + sliding-window memory as the notebook, but retrieval is
    done outside so we can show the retrieved chunks in the Sources panel."""
    global _ui_chain
    if _ui_chain is None:
        from langchain_core.runnables.history import RunnableWithMessageHistory
        from langchain_core.output_parsers import StrOutputParser
        _ui_chain = RunnableWithMessageHistory(
            qa_prompt | llm | StrOutputParser(),
            get_session_history,
            input_messages_key="input",
            history_messages_key="chat_history",
        )
    return _ui_chain

def retrieve(question):
    docs = hybrid_retriever.invoke(question)[:TOP_K]
    sources = []
    for d in docs:
        sources.append({
            "title": (d.page_content or "")[:90].replace("\n", " "),
            "snippet": (d.metadata.get("answer", "") or "")[:420],
            "score": d.metadata.get("score_answer", "—"),
        })
    return format_docs(docs), sources

def stream_answer(question, context, sid):
    chain = _get_chain()
    cfg = {"configurable": {"session_id": sid}}
    for chunk in chain.stream({"input": question, "context": context}, config=cfg):
        yield chunk if isinstance(chunk, str) else getattr(chunk, "content", "") or ""

def seed_memory(sid):
    """After a restart, give the LLM memory of a saved chat again."""
    from langchain_core.messages import HumanMessage, AIMessage
    h = get_session_history(sid)
    if not h.messages:
        for m in SESSIONS[sid]["messages"][-10:]:
            h.add_message(HumanMessage(content=m["content"]) if m["role"] == "user" else AIMessage(content=m["content"]))

# ------------------------------------------------------------ chat storage --
def _load():
    try:
        with open(HISTORY_FILE, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}

def _save():
    try:
        with open(HISTORY_FILE, "w", encoding="utf-8") as f:
            json.dump(SESSIONS, f, ensure_ascii=False)
    except Exception as e:
        print("could not save history:", e)

SESSIONS = _load()   # {sid: {"title", "ts", "messages", "sources"}}

def _choices():
    items = sorted(SESSIONS.items(), key=lambda kv: kv[1].get("ts", 0), reverse=True)
    return [(v.get("title") or "New chat", k) for k, v in items]

def _hist(sid=None):
    return gr.update(choices=_choices(), value=sid if sid in SESSIONS else None)

def render_sources(sources):
    if not sources:
        return "### Retrieved context\n\nAsk a question and the chunks the model used will appear here."
    out = ["### Retrieved context", f"_{len(sources)} chunks · hybrid BM25 + dense · RRF ranked_", ""]
    for i, s in enumerate(sources, 1):
        out += [f"**{i:02d} · {s['title']}**", f"`answer votes: {s['score']}`", f"> {s['snippet']}", ""]
    return "\n".join(out)

def _src_label(sources):
    return f"Sources · {len(sources)}" if sources else "Sources"

# ------------------------------------------------------------- UI callbacks --
def submit(question, chat, sid):
    q = (question or "").strip()
    if not q:
        yield gr.update(), "", sid, gr.update(), gr.update(), gr.update(), gr.update()
        return
    if not sid or sid not in SESSIONS:
        sid = uuid.uuid4().hex[:8]
        SESSIONS[sid] = {"title": q[:44] + ("…" if len(q) > 44 else ""), "ts": time.time(), "messages": [], "sources": []}
    sess = SESSIONS[sid]
    chat = list(chat or []) + [{"role": "user", "content": q}, {"role": "assistant", "content": "Thinking…"}]
    sources = []
    yield chat, "", sid, gr.update(visible=False), gr.update(), _hist(sid), gr.update()

    if not _pipeline_ready():
        chat[-1]["content"] = ("**The RAG pipeline is not loaded in this notebook.** "
                               "Run the RAG cells first (retriever, prompt, LLM, memory), then run this cell again.")
        yield chat, "", sid, gr.update(), gr.update(), gr.update(), gr.update()
        return
    try:
        context, sources = retrieve(q)
        yield chat, "", sid, gr.update(), render_sources(sources), gr.update(), gr.update(value=_src_label(sources))
        seed_memory(sid)
        text = ""
        for piece in stream_answer(q, context, sid):
            text += piece
            # avoid showing the cursor mid code-fence (odd number of ``` = inside a block)
            fence_count = text.count("```")
            cursor = "" if fence_count % 2 == 1 else " ▍"
            chat[-1]["content"] = text + cursor
            yield chat, "", sid, gr.update(), gr.update(), gr.update(), gr.update()
        chat[-1]["content"] = text or "_(empty answer)_"
    except Exception as e:
        chat[-1]["content"] = f"**Backend error:** `{type(e).__name__}: {e}`"
    sess["messages"], sess["sources"], sess["ts"] = chat, sources, time.time()
    _save()
    yield chat, "", sid, gr.update(), gr.update(), _hist(sid), gr.update()

def load_session(sid):
    s = SESSIONS.get(sid)
    if not s:
        return gr.update(), sid, gr.update(), gr.update(), gr.update()
    return s["messages"], sid, gr.update(visible=not s["messages"]), render_sources(s["sources"]), gr.update(value=_src_label(s["sources"]))

def new_chat():
    return [], "", gr.update(visible=True), render_sources([]), gr.update(value=_src_label([])), _hist(None)

def delete_chat(sid):
    SESSIONS.pop(sid, None)
    try:
        get_session_history(sid).clear()
    except Exception:
        pass
    _save()
    return new_chat()

# --------------------------------------------------------- architecture SVG --
def build_arch_svg():
    W, H, NW, NH = 1660, 560, 196, 86
    N = {  # name: (x, y, title, line1, line2, badge, kind)
        "data":  (10,   236, "Dataset",         "instruct-python-500k",   "100K sampled Q&amp;A",       "1",  ""),
        "clean": (246,  236, "Clean &amp; Parse",   "HTML unescape + regex",  "Document objects",     "2",  ""),
        "dense": (482,  104, "Chroma Dense",    "MiniLM-L6 embeddings",   "top-10 by similarity",   "3a", ""),
        "bm25":  (482,  368, "BM25 Sparse",     "keyword index",          "top-10 by term match",   "3b", ""),
        "rrf":   (718,  236, "RRF Ensemble", "Reciprocal Rank Fusion", "best 5 chunks",         "4",  "hub"),
        "prompt":(954,  236, "Prompt Builder",  "rules + context",        "+ chat history",          "5",  ""),
        "llm":   (1190, 236, "Qwen LLM (Local)",     "grounded reasoning",    "adapts, never copies",   "6",  "hub"),
        "out":   (1426, 236, "Answer",          "streamed to the chat",   "+ sources panel",         "7",  ""),
        "query": (718,  34,  "User Query",      "question + session_id",  "",                        "Q",  "ghost"),
        "mem":   (954,  430, "Sliding Memory",  "last 10 messages",       "per session",             "M",  "ghost"),
    }
    def rc(n): x, y = N[n][:2]; return x + NW, y + NH / 2
    def lc(n): x, y = N[n][:2]; return x, y + NH / 2
    def tc(n): x, y = N[n][:2]; return x + NW / 2, y
    def bc(n): x, y = N[n][:2]; return x + NW / 2, y + NH
    def path(p, q, mode="h"):
        (px, py), (qx, qy) = p, q
        if mode == "h":
            mx = (px + qx) / 2; return f"M{px} {py} C{mx} {py} {mx} {qy} {qx} {qy}"
        if mode == "v":
            my = (py + qy) / 2; return f"M{px} {py} C{px} {my} {qx} {my} {qx} {qy}"
        if mode == "hv": return f"M{px} {py} Q{qx} {py} {qx} {qy}"
        return f"M{px} {py} Q{px} {qy} {qx} {qy}"
    edges = [
        (rc("data"), lc("clean"), "h", 0), (rc("clean"), lc("dense"), "h", 0), (rc("clean"), lc("bm25"), "h", 0),
        (rc("dense"), lc("rrf"), "h", 0), (rc("bm25"), lc("rrf"), "h", 0), (rc("rrf"), lc("prompt"), "h", 0),
        (rc("prompt"), lc("llm"), "h", 0), (rc("llm"), lc("out"), "h", 0),
        (bc("query"), tc("rrf"), "v", 1), (rc("query"), tc("prompt"), "hv", 1),
        (tc("mem"), bc("prompt"), "v", 1), (bc("out"), rc("mem"), "vh", 1),
    ]
    s = [f'<svg viewBox="0 0 {W} {H}" xmlns="http://www.w3.org/2000/svg" class="arch-svg" role="img" aria-label="PyRexa RAG pipeline architecture">',
         f'<defs><linearGradient id="ag" gradientUnits="userSpaceOnUse" x1="0" y1="0" x2="{W}" y2="0"><stop offset="0" stop-color="#17b7a9"/><stop offset="1" stop-color="#36e8e1"/></linearGradient>'
         '<filter id="glow" x="-30%" y="-30%" width="160%" height="160%"><feGaussianBlur stdDeviation="6" result="b"/><feMerge><feMergeNode in="b"/><feMergeNode in="SourceGraphic"/></feMerge></filter></defs>']
    for i, (p, q, mode, soft) in enumerate(edges):
        s.append(f'<path id="e{i}" d="{path(p, q, mode)}" class="arch-edge{" soft" if soft else ""}"/>')
    for i, (p, q, mode, soft) in enumerate(edges):
        if soft: continue
        s.append(f'<circle r="4.5" class="arch-dot"><animateMotion dur="{2.6 + (i % 3) * .5}s" begin="-{(i % 4) * .7}s" repeatCount="indefinite"><mpath href="#e{i}"/></animateMotion></circle>')
    for name, (x, y, t, l1, l2, badge, kind) in N.items():
        s.append(f'<g class="anode {kind}"><rect x="{x}" y="{y}" width="{NW}" height="{NH}" rx="22"/>'
                 f'<circle cx="{x+30}" cy="{y+NH/2}" r="16" class="abadge"/><text x="{x+30}" y="{y+NH/2+4.5}" class="abt">{badge}</text>'
                 f'<text x="{x+54}" y="{y+34}" class="at">{t}</text><text x="{x+54}" y="{y+54}" class="as">{l1}</text><text x="{x+54}" y="{y+69}" class="as">{l2}</text></g>')
    s.append(f'<text x="{(10+246+NW)/2}" y="548" class="agrp">INDEXING</text><text x="{(482+718+NW)/2}" y="548" class="agrp">RETRIEVAL</text>'
             f'<text x="{(954+1426+NW)/2}" y="548" class="agrp">GENERATION</text>')
    s.append('<text x="1030" y="392" class="alabel">history</text><text x="1310" y="484" class="alabel">save turn</text>')
    s.append('</svg>')
    return "".join(s)

# -------------------------------------------------------------------- style --
CSS = """
:root,.dark,.gradio-container{
 --body-background-fill:#05080c;--background-fill-primary:#0a1016;--background-fill-secondary:#0d151c;
 --block-background-fill:transparent;--block-border-width:0px;--block-shadow:none;--body-text-color:#e9fbfa;
 --body-text-color-subdued:#8fa6a8;--border-color-primary:rgba(54,232,225,.16);--input-background-fill:transparent;
 --input-border-width:0px;--input-shadow:none;--input-shadow-focus:none;--color-accent:#17b7a9;
 --button-primary-background-fill:linear-gradient(120deg,#0ea99f,#36e8e1);--button-primary-text-color:#031718;
 --button-secondary-background-fill:rgba(255,255,255,.04);--button-secondary-text-color:#d9f7f5;--button-secondary-border-color:rgba(54,232,225,.22);
 --radius-lg:18px;--radius-xl:24px}
html,body,.gradio-container{background:#05080c!important;color:#e9fbfa!important;font-family:'Inter','Segoe UI',system-ui,sans-serif!important}
.gradio-container{max-width:100%!important;padding:0!important}footer{display:none!important}
::-webkit-scrollbar{width:8px;height:8px}::-webkit-scrollbar-thumb{background:#1f5357;border-radius:20px}
@keyframes spin{to{transform:rotate(360deg)}}@keyframes float{0%,100%{transform:translateY(0)}50%{transform:translateY(-10px)}}
@keyframes pulse{0%,100%{box-shadow:0 0 28px #36e8e144,0 0 90px #17b7a933}50%{box-shadow:0 0 44px #36e8e199,0 0 140px #17b7a955}}
@keyframes dash{to{stroke-dashoffset:-28}}@keyframes rise{from{opacity:0;transform:translateY(18px)}to{opacity:1;transform:none}}

/* ---------- landing ---------- */
#landing{position:relative;min-height:100vh;overflow:hidden;gap:0!important;padding:0!important;
 background:radial-gradient(circle at 78% 18%,#0e6f6a38,transparent 38%),radial-gradient(circle at 8% 88%,#0a8f9a26,transparent 40%),#05080c}
#landing:before{content:"";position:absolute;inset:0;pointer-events:none;background-image:linear-gradient(rgba(54,232,225,.045) 1px,transparent 1px),linear-gradient(90deg,rgba(54,232,225,.045) 1px,transparent 1px);background-size:58px 58px;-webkit-mask-image:radial-gradient(ellipse at 50% 25%,#000 15%,transparent 72%);mask-image:radial-gradient(ellipse at 50% 25%,#000 15%,transparent 72%)}
.lp-nav{position:relative;z-index:2;display:flex;align-items:center;justify-content:space-between;max-width:1360px;margin:0 auto;padding:26px 28px}
.lp-nav img{height:40px;width:auto}.lp-pill{border:1px solid rgba(54,232,225,.25);border-radius:999px;padding:9px 16px;color:#9fdcd8;font-size:11px;letter-spacing:.16em;text-transform:uppercase;background:rgba(54,232,225,.05)}
.lp-hero{position:relative;z-index:2;max-width:1360px;margin:0 auto;padding:30px 28px 10px;display:grid;grid-template-columns:1.05fr .95fr;gap:4vw;align-items:center}
.lp-eyebrow{color:#36e8e1;letter-spacing:.22em;font-size:12px;font-weight:800;text-transform:uppercase}
.lp-hero h1{font-size:clamp(40px,5vw,78px);line-height:1;letter-spacing:-.06em;margin:22px 0;background:linear-gradient(120deg,#fff 5%,#9ff7ee 50%,#20c9c1 95%);-webkit-background-clip:text;background-clip:text;color:transparent;animation:rise .9s both}
.lp-hero p{color:#93a9ab;font-size:clamp(16px,1.4vw,20px);line-height:1.75;max-width:600px}
.lp-stats{display:flex;flex-wrap:wrap;gap:10px;margin-top:26px}.lp-stats span{border:1px solid rgba(54,232,225,.2);background:rgba(54,232,225,.06);color:#bff5f0;border-radius:999px;padding:8px 14px;font-size:12px}
.neon-stage{position:relative;display:flex;flex-direction:column;align-items:center}
.neon-wrap{position:relative;width:min(100%,520px);border-radius:56px;padding:2px;overflow:hidden;animation:pulse 3.8s ease-in-out infinite,float 7s ease-in-out infinite}
.neon-wrap:before{content:"";position:absolute;inset:-70%;background:conic-gradient(from 0deg,#36e8e1,transparent 22%,#17b7a9 48%,transparent 72%,#36e8e1);animation:spin 7s linear infinite}
.neon-card{position:relative;border-radius:54px;padding:46px 34px;background:radial-gradient(circle at 50% 35%,#123a3f,#081118 72%);display:flex;justify-content:center}
.neon-card img{width:88%;height:auto;filter:drop-shadow(0 0 14px #36e8e1aa) drop-shadow(0 0 40px #17b7a988)}
.neon-floor{width:72%;height:38px;margin-top:34px;background:radial-gradient(ellipse,#36e8e177,transparent 70%);filter:blur(12px)}
.lp-cta{position:relative;z-index:2;max-width:1360px;margin:8px auto 0!important;padding:0 28px!important;gap:14px!important}
#enter-btn{max-width:290px}#enter-btn,#enter-btn button{border-radius:999px!important;font-weight:850!important;font-size:16px!important;min-height:54px;box-shadow:0 14px 40px #10b9b544,0 0 0 1px #ffffff14 inset!important;transition:transform .2s,filter .2s}
#enter-btn:hover{transform:translateY(-2px);filter:brightness(1.1)}
.lp-arch{position:relative;z-index:2;max-width:1360px;margin:34px auto 60px;padding:0 28px}
.lp-arch h2{font-size:clamp(26px,3vw,38px);letter-spacing:-.04em;margin:0 0 6px}.lp-arch .sub{color:#8fa6a8;margin:0 0 20px}
.arch-scroll{overflow-x:auto;border:1px solid rgba(54,232,225,.18);border-radius:30px;background:linear-gradient(160deg,rgba(16,32,38,.85),rgba(8,14,19,.9));padding:18px;box-shadow:0 30px 80px #0009,inset 0 1px 0 #ffffff08}
.arch-svg{min-width:1040px;width:100%;height:auto;display:block}
.arch-edge{fill:none;stroke:url(#ag);stroke-width:2.2;stroke-dasharray:6 8;animation:dash 1.5s linear infinite}.arch-edge.soft{stroke:#5ddad3;opacity:.45;stroke-width:1.8}
.arch-dot{fill:#7ffcf3;filter:drop-shadow(0 0 6px #36e8e1)}
.anode rect{fill:#0c1b21;stroke:rgba(54,232,225,.42);stroke-width:1.4;transition:.25s}.anode.hub rect{fill:#0f2f33;stroke:#36e8e1;filter:url(#glow)}
.anode.ghost rect{fill:#0a1418;stroke-dasharray:5 5;stroke:rgba(54,232,225,.35)}.anode:hover rect{stroke:#7ffcf3;fill:#123a3f}
.abadge{fill:#0e5b5b;stroke:#36e8e1;stroke-width:1.2}.abt{fill:#dffffb;font-size:11px;font-weight:800;text-anchor:middle}
.at{fill:#f0fffe;font-size:15px;font-weight:800}.as{fill:#8fb2b3;font-size:11px}.agrp{fill:#37a9a3;font-size:12px;letter-spacing:.3em;font-weight:800;text-anchor:middle}.alabel{fill:#5ba8a4;font-size:11px;font-style:italic}

/* ---------- workspace ---------- */
#workspace{position:relative;min-height:100vh;gap:0!important;padding:0!important;background:radial-gradient(circle at 70% -5%,#0f5f5d2e,transparent 34%),radial-gradient(circle at 0% 100%,#0b88962b,transparent 36%),#05080c}
#topbar{position:fixed!important;top:0;left:0;right:0;z-index:70;padding:10px 18px!important;flex-wrap:nowrap!important;align-items:center;gap:10px!important;background:linear-gradient(180deg,#05080cf5 55%,#05080c00)}
#topbar .icon-btn{flex:0 0 auto!important;white-space:nowrap!important;padding:0 16px!important;width:auto!important}
#topbar .tb-center{flex:1 1 auto!important}
.icon-btn,.icon-btn button{border-radius:14px!important;min-height:42px!important;background:rgba(255,255,255,.045)!important;border:1px solid rgba(54,232,225,.2)!important;color:#d8f8f5!important;font-weight:650!important}
.icon-btn:hover{border-color:#36e8e1!important;box-shadow:0 0 22px #36e8e122}
.tb-brand{display:flex;align-items:center;justify-content:center;gap:10px;font-weight:800;letter-spacing:-.02em}.tb-brand img{height:30px;filter:drop-shadow(0 0 8px #36e8e177)}.tb-brand small{color:#7f9a9c;font-weight:500}
.drawer{position:fixed!important;top:66px;bottom:14px;width:318px;z-index:60;padding:18px!important;overflow:auto;gap:10px!important;
 background:rgba(9,16,22,.93)!important;backdrop-filter:blur(20px);border:1px solid rgba(54,232,225,.24)!important;border-radius:26px!important;box-shadow:0 26px 80px #000c,0 0 40px #36e8e112!important;
 opacity:0;pointer-events:none;transition:transform .38s cubic-bezier(.2,.8,.2,1),opacity .3s}
.drawer-left{left:14px;transform:translateX(-118%)}.drawer-right{right:14px;width:390px;transform:translateX(118%)}
.drawer.open{transform:none!important;opacity:1!important;pointer-events:auto!important}
.drawer h3{margin:0 0 4px;font-size:15px;letter-spacing:.02em}
.hist{background:transparent!important}.hist input[type=radio]{display:none!important}.hist .wrap{display:flex!important;flex-direction:column!important;gap:6px!important}
.hist label{display:block!important;width:100%;padding:12px 14px!important;border-radius:14px!important;border:1px solid transparent!important;background:transparent!important;color:#cfeeed!important;cursor:pointer;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.hist label:hover{background:rgba(54,232,225,.07)!important}.hist label.selected{background:rgba(54,232,225,.13)!important;border-color:rgba(54,232,225,.3)!important}
.sources-md{font-size:13px;line-height:1.6}.sources-md code{background:rgba(54,232,225,.12)!important;color:#aef7f0!important;border-radius:8px;padding:2px 8px;border:0!important}.sources-md blockquote{border-left:2px solid #36e8e1;background:rgba(54,232,225,.06);border-radius:0 12px 12px 0;padding:8px 12px;color:#9fb6b8;margin:6px 0 14px}
#stage{position:relative;z-index:1;width:100%;max-width:880px;margin:0 auto!important;padding:78px 18px 130px!important;min-height:100vh;gap:8px!important}
#welcome{align-items:center;text-align:center;padding:5vh 0 0!important;gap:14px!important}
.hello-orb{width:84px;height:84px;margin:0 auto 6px;border-radius:26px;display:flex;align-items:center;justify-content:center;background:radial-gradient(circle,#123f42,#081118);border:1px solid rgba(54,232,225,.45);box-shadow:0 0 40px #36e8e155,0 0 90px #17b7a933;animation:float 6s ease-in-out infinite}
.hello-orb img{width:58px}
.hello h2{font-size:clamp(30px,4.4vw,50px);letter-spacing:-.05em;margin:8px 0 6px;background:linear-gradient(120deg,#fff,#8ff5ec 55%,#25c9c1);-webkit-background-clip:text;background-clip:text;color:transparent}.hello p{color:#8fa6a8;margin:0}
#chips{gap:10px!important;flex-wrap:wrap!important;justify-content:center;margin-top:10px}
.chip,.chip button{border-radius:18px!important;background:rgba(255,255,255,.04)!important;border:1px solid rgba(54,232,225,.2)!important;color:#cdf3f0!important;text-align:left;font-size:13.5px!important;padding:12px 16px!important;min-height:52px;transition:.2s}
.chip:hover{transform:translateY(-3px);border-color:#36e8e1!important;background:rgba(54,232,225,.09)!important}
#chat{background:transparent!important;border:0!important;box-shadow:none!important;height:calc(100vh - 215px)!important;min-height:280px}
#chat .bubble-wrap,#chat .wrap,#chat>div{background:transparent!important;border:0!important}
#chat .message-row,#chat .message-wrap,#chat .bubble-wrap{background:transparent!important;border:0!important;box-shadow:none!important}
#chat .message,#chat [data-testid=bot],#chat [data-testid=user]{font-size:15.5px!important;line-height:1.75!important}
#chat .message.user{background:linear-gradient(135deg,#17323a,#122029)!important;border:1px solid rgba(54,232,225,.24)!important;border-radius:22px 22px 6px 22px!important;color:#eafffd!important}
#chat .message.bot{background:transparent!important;border:0!important;box-shadow:none!important;color:#e6f7f6!important}
#chat .message .message,#chat .message [data-testid]{background:transparent!important;border:0!important}
#chat code:not(pre code){background:rgba(54,232,225,.1)!important;color:#aef7f0!important;padding:2px 6px}
#chat pre,#chat code{border-radius:14px!important}#chat pre{background:#070d12!important;border:1px solid rgba(54,232,225,.16)!important}
#chat h1,#chat h2,#chat h3{color:#eafffd;font-weight:800;margin:14px 0 8px;letter-spacing:-.02em}
#chat h3{font-size:17px}
#chat hr{border:0;border-top:1px solid rgba(54,232,225,.2);margin:14px 0}
#chat ul,#chat ol{padding-inline-start:22px;margin:8px 0}
#chat li{margin:4px 0}
#chat strong{color:#9ff7ee}
#chat p{margin:8px 0}
#chat blockquote{border-left:2px solid #36e8e1;background:rgba(54,232,225,.05);border-radius:0 10px 10px 0;padding:6px 12px;margin:8px 0;color:#a9c5c6}
#chat .avatar-container img,#chat .avatar-image{border-radius:50%!important;background:radial-gradient(circle,#123f42,#081118);border:1px solid rgba(54,232,225,.5);box-shadow:0 0 18px #36e8e166;padding:3px}
#composer{position:fixed!important;bottom:34px;left:50%;transform:translateX(-50%);width:min(844px,calc(100% - 32px))!important;z-index:20;flex-wrap:nowrap!important;align-items:center;gap:8px!important;padding:8px 10px 8px 22px!important;border-radius:32px!important;
 background:rgba(13,21,28,.94)!important;backdrop-filter:blur(16px);border:1px solid rgba(54,232,225,.3)!important;box-shadow:0 14px 55px #000b,0 0 44px #36e8e114!important}
#composer:focus-within{border-color:#36e8e1!important;box-shadow:0 14px 55px #000b,0 0 50px #36e8e13a!important}
#composer textarea{background:transparent!important;border:0!important;box-shadow:none!important;font-size:16px!important;padding:12px 0!important}
#send,#send button{min-width:50px!important;width:50px;height:50px;border-radius:50%!important;padding:0!important;box-shadow:0 8px 26px #10b9b555!important}
.fine{position:fixed;bottom:8px;left:0;right:0;z-index:20;color:#5d7476;font-size:11.5px;text-align:center;margin:0}
#stage:has(#welcome) #chat{display:none!important}
#chat .message-row.bot-row,#chat .message.bot{width:100%!important;max-width:100%!important}
@media(max-width:900px){.lp-hero{grid-template-columns:1fr}.neon-stage{order:-1}.drawer{width:86vw!important}#chat{height:calc(100vh - 250px)!important}}

/* ---------- teal accent everywhere (replaces Gradio's default orange) ---------- */
:root,.dark,body,.gradio-container,.gradio-container.dark{
 --primary-50:#e6fffd!important;--primary-100:#c2fbf7!important;--primary-200:#8ff5ee!important;--primary-300:#5aece3!important;
 --primary-400:#36e8e1!important;--primary-500:#17b7a9!important;--primary-600:#0e9a90!important;--primary-700:#0c7a73!important;
 --primary-800:#0d615c!important;--primary-900:#0e504c!important;--primary-950:#052f2d!important;
 --color-accent:#17b7a9!important;--color-accent-soft:rgba(54,232,225,.14)!important;
 --border-color-accent:#36e8e1!important;--border-color-accent-subdued:rgba(54,232,225,.35)!important;
 --input-border-color-focus:#36e8e1!important;--input-border-color-hover:rgba(54,232,225,.4)!important;
 --loader-color:#36e8e1!important;--slider-color:#17b7a9!important;
 --checkbox-background-color-selected:#17b7a9!important;--checkbox-border-color-selected:#36e8e1!important;--checkbox-border-color-focus:#36e8e1!important;
 --link-text-color:#36e8e1!important;--link-text-color-hover:#7ffcf3!important;--link-text-color-active:#36e8e1!important;--link-text-color-visited:#5ddad3!important;
 --button-primary-background-fill-hover:linear-gradient(120deg,#12c2b7,#63f3ec)!important;
 --button-primary-border-color:transparent!important;--button-primary-border-color-hover:transparent!important;
 --button-primary-text-color-hover:#031718!important;
 --button-secondary-background-fill-hover:rgba(54,232,225,.1)!important;--button-secondary-border-color-hover:#36e8e1!important;
 --button-secondary-text-color-hover:#eafffd!important}
.gradio-container *{-webkit-tap-highlight-color:transparent}
.gradio-container button:focus,.gradio-container button:focus-visible,.gradio-container textarea:focus,.gradio-container input:focus{outline:none!important}
.gradio-container button:focus-visible{box-shadow:0 0 0 2px rgba(54,232,225,.55)!important}
.gradio-container button:active{filter:brightness(.92);transform:scale(.97)}
.icon-btn:active,.icon-btn button:active{background:rgba(54,232,225,.16)!important;border-color:#36e8e1!important;color:#fff!important}
.chip:active,.chip button:active{background:rgba(54,232,225,.16)!important;border-color:#36e8e1!important}
.generating,.pending,.wrap.generating{border-color:transparent!important;animation:none!important;box-shadow:none!important}
.eta-bar{background:rgba(23,183,169,.12)!important}

/* ---------- SVG icon system (no emojis) ---------- */
.ico-btn{display:inline-flex!important;align-items:center;justify-content:center;gap:9px}
.ico-btn::before,.ico-btn.ico-r::after{content:"";display:inline-block;width:19px;height:19px;flex:0 0 19px;background:currentColor;
 -webkit-mask:var(--ic) center/contain no-repeat;mask:var(--ic) center/contain no-repeat}
.ico-btn.ico-r::before{display:none}
.ico-only{gap:0!important}
#topbar .ico-only{padding:0 13px!important}
.ico-only::before{width:21px;height:21px;flex:0 0 21px}
#send.ico-only::before{width:24px;height:24px;flex:0 0 24px}
.icon-btn.ico-btn:hover::before{filter:drop-shadow(0 0 6px #36e8e1aa)}
"""

# --- icon paths (24x24, heavy rounded stroke) -> CSS mask variables ---------
def _ico(body, sw=2.6):
    svg = (f"<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' stroke='#000' "
           f"stroke-width='{sw}' stroke-linecap='round' stroke-linejoin='round'>{body}</svg>")
    return 'url("data:image/svg+xml,' + quote(svg) + '")'

ICONS = {
    "history": "<path d='M3 12a9 9 0 1 0 3-6.7'/><path d='M3 4v5h5'/><path d='M12 7v5l3.2 2'/>",
    "layers":  "<path d='M12 3 3 8l9 5 9-5-9-5z'/><path d='m3 12.5 9 5 9-5'/><path d='m3 17 9 5 9-5'/>",
    "plus":    "<path d='M12 5v14M5 12h14'/>",
    "home":    "<path d='M3 11.5 12 3.5l9 8'/><path d='M5.5 10v10h4.5v-6h4v6h4.5V10'/>",
    "send":    "<path d='M12 19.5V5'/><path d='m5.5 11.5 6.5-6.5 6.5 6.5'/>",
    "trash":   "<path d='M4 7h16'/><path d='M9 7V4h6v3'/><path d='M6 7l1 13h10l1-13'/><path d='M10 11v5M14 11v5'/>",
    "arrow":   "<path d='M4.5 12h15'/><path d='m13 5.5 6.5 6.5-6.5 6.5'/>",
    "close":   "<path d='M6 6l12 12M18 6 6 18'/>",
}
CSS += "".join(f".i-{k}{{--ic:{_ico(v)}}}" for k, v in ICONS.items())
_BLANK = "\u200b"   # empty label for icon-only buttons

JS_LEFT = "()=>{const l=document.querySelector('.drawer-left'),r=document.querySelector('.drawer-right');r&&r.classList.remove('open');l&&l.classList.toggle('open')}"
JS_RIGHT = "()=>{const l=document.querySelector('.drawer-left'),r=document.querySelector('.drawer-right');l&&l.classList.remove('open');r&&r.classList.toggle('open')}"
JS_CLOSE = "()=>{document.querySelectorAll('.drawer').forEach(d=>d.classList.remove('open'))}"

CHIPS = [
    "How do I reverse a list of tuples without reversed()?",
    "Difference between a list and a tuple?",
    "Read a CSV with pandas and group by a column",
    "Fix: 'NoneType' object is not subscriptable",
]

_blocks_kw, _launch_kw = {"title": "PyRexa"}, {}
if "fill_width" in inspect.signature(gr.Blocks.__init__).parameters:
    _blocks_kw["fill_width"] = True
if "css" in inspect.signature(gr.Blocks.__init__).parameters:
    _blocks_kw["css"] = CSS
else:
    _launch_kw["css"] = CSS

_chat_kw = {"show_label": False, "elem_id": "chat", "avatar_images": (None, AVATAR_PATH)}
_chat_kw["render_markdown"] = True
_cp = inspect.signature(gr.Chatbot.__init__).parameters
if "type" in _cp: _chat_kw["type"] = "messages"

# ---------------------------------------------------------------------- app --
with gr.Blocks(**_blocks_kw) as app:
    sid_state = gr.State("")

    # ===== landing page =====
    with gr.Column(visible=True, elem_id="landing") as landing:
        gr.HTML(f"""
        <div class='lp-nav'><img src='{LOGO_DATA}' alt='PyRexa'><span class='lp-pill'>Python RAG assistant</span></div>
        <div class='lp-hero'>
          <div>
            <div class='lp-eyebrow'>PyRexa · Grounded Python answers</div>
            <h1>Ask Python.<br>Get answers<br>you can trust.</h1>
            <p>PyRexa searches 100K real Stack Overflow Q&amp;As with hybrid retrieval (semantic + keyword), then lets Gemini adapt the best evidence to your exact code.</p>
            <div class='lp-stats'><span>100K Q&amp;A knowledge base</span><span>BM25 + Dense + RRF</span><span>Qwen generation</span><span>Session memory</span></div>
          </div>
          <div class='neon-stage'>
            <div class='neon-wrap'><div class='neon-card'><img src='{LOGO_DATA}' alt='PyRexa logo'></div></div>
            <div class='neon-floor'></div>
          </div>
        </div>""")
        with gr.Row(elem_classes="lp-cta"):
            enter_btn = gr.Button("Enter the workspace", variant="primary", elem_id="enter-btn",
                                  elem_classes=["ico-btn", "ico-r", "i-arrow"], scale=0)
        gr.HTML(f"""
        <div class='lp-arch'>
          <h2>How PyRexa thinks</h2>
          <p class='sub'>The full pipeline behind every answer — from raw data to a grounded reply.</p>
          <div class='arch-scroll'>{build_arch_svg()}</div>
        </div>""")

    # ===== chat workspace =====
    with gr.Column(visible=False, elem_id="workspace") as workspace:
        with gr.Row(elem_id="topbar"):
            btn_menu = gr.Button("History", elem_classes=["icon-btn", "ico-btn", "i-history"], scale=0, min_width=120)
            gr.HTML(f"<div class='tb-brand'><img src='{LOGO_DATA}'><span>PyRexa <small>/ workspace</small></span></div>", elem_classes="tb-center")
            btn_src = gr.Button("Sources", elem_classes=["icon-btn", "ico-btn", "i-layers"], scale=0, min_width=130)
            btn_new = gr.Button(_BLANK, elem_classes=["icon-btn", "ico-btn", "ico-only", "i-plus"], scale=0, min_width=48)
            btn_home = gr.Button(_BLANK, elem_classes=["icon-btn", "ico-btn", "ico-only", "i-home"], scale=0, min_width=48)

        with gr.Column(elem_classes="drawer drawer-left"):
            gr.HTML("<h3>Chat history</h3><div style='color:#7f9a9c;font-size:12px'>Saved automatically</div>")
            btn_new2 = gr.Button("New chat", variant="primary", elem_classes=["ico-btn", "i-plus"])
            hist = gr.Radio(choices=_choices(), value=None, show_label=False, container=False, elem_classes="hist")
            btn_del = gr.Button("Delete this chat", size="sm", elem_classes=["icon-btn", "ico-btn", "i-trash"])
            btn_close_l = gr.Button("Close", size="sm", elem_classes=["icon-btn", "ico-btn", "i-close"])

        with gr.Column(elem_classes="drawer drawer-right"):
            sources_md = gr.Markdown(render_sources([]), elem_classes="sources-md")
            btn_close_r = gr.Button("Close", size="sm", elem_classes=["icon-btn", "ico-btn", "i-close"])

        with gr.Column(elem_id="stage"):
            with gr.Column(elem_id="welcome") as welcome:
                gr.HTML(f"<div class='hello'><div class='hello-orb'><img src='{AVATAR_DATA}'></div><h2>Hello, what are we building today?</h2><p>Ask anything about Python — I answer from real, retrieved examples.</p></div>")
                with gr.Row(elem_id="chips"):
                    chip_btns = [gr.Button(c, elem_classes="chip", scale=0, min_width=260) for c in CHIPS]
            chat = gr.Chatbot(**_chat_kw)
            with gr.Row(elem_id="composer"):
                msg = gr.Textbox(placeholder="Ask PyRexa anything about Python…", lines=1, max_lines=6, show_label=False, container=False, scale=10)
                send = gr.Button(_BLANK, variant="primary", elem_id="send", elem_classes=["ico-btn", "ico-only", "i-send"], scale=0, min_width=50)
            gr.HTML("<p class='fine'>PyRexa can make mistakes — double-check important code before using it.</p>")

    # ===== wiring =====
    outs = [chat, msg, sid_state, welcome, sources_md, hist, btn_src]
    enter_btn.click(lambda: (gr.update(visible=False), gr.update(visible=True)), outputs=[landing, workspace])
    btn_home.click(lambda: (gr.update(visible=True), gr.update(visible=False)), outputs=[landing, workspace], js=JS_CLOSE)
    btn_menu.click(None, None, None, js=JS_LEFT)
    btn_src.click(None, None, None, js=JS_RIGHT)
    btn_close_l.click(None, None, None, js=JS_CLOSE)
    btn_close_r.click(None, None, None, js=JS_CLOSE)

    msg.submit(submit, [msg, chat, sid_state], outs)
    send.click(submit, [msg, chat, sid_state], outs)
    for _b, _t in zip(chip_btns, CHIPS):
        _b.click(submit, [gr.State(_t), chat, sid_state], outs)

    new_outs = [chat, sid_state, welcome, sources_md, btn_src, hist]
    btn_new.click(new_chat, None, new_outs)
    btn_new2.click(new_chat, None, new_outs).then(None, None, None, js=JS_CLOSE)
    btn_del.click(delete_chat, [sid_state], new_outs)
    hist.input(load_session, [hist], [chat, sid_state, welcome, sources_md, btn_src]).then(None, None, None, js=JS_CLOSE)

app.queue().launch(share=True, debug=True, **_launch_kw)
# ============================================================================

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://00f8514d777ab48e22.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_1987/2853908684.py:71: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain = _get_chain()
